In [ ]:
!pip install ipympl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from functools import partial
from scipy.fft import fft, fftshift, fftfreq, rfft

In [ ]:
"""
This cell is a cousin to the animation from Exercise 1, but without the phasor.
You ARE required to understand it, as you may be modifying it later.
Read and fill in the simulation code (next cell) first, to better understand this one.
"""
def animate_1D(ks, xs, wave_list, omegas): # call this function to make your animation
    def update(frame, lines): # an inner function for the purpose of preparing the next frame
        # wave_list contains a dictionary of 
        if frame==0:
            return
        _totals = {
            field:np.sum(
                wave_list(field)*np.exp(1j*ks*xs-1j*omegas*frame*delta_t) # propagate each wave, in each field, and add them together
            ) for field in wave_list.keys() # sums up actual contribution each each wave to each field at (x,t)
        }
        for field in wave_list.keys():
            lines[field].set_ydata(np.real(_totals[field])) # change the y-data on our plot for each plotting element
        return (*lines.values,) # change and return these elements to the main animation function


    fig, axes = plt.subplots(2) # can change no. of subplots
    for ax in axes:
        ax.set_xlabel(r"$x$")
    _totals = {key:np.sum(wave_list(key)*np.exp(1j*ks*xs)) for key in wave_list.keys()}
    lines = {}
    lines["u"] = axes[0].plot(xs, np.real(_totals["u"]))[0] # We have extracted the plotting element encoding our line plot
    lines["eta"] = axes[1].plot(xs, np.real(_totals["eta"]))[0] # and here we have a second such element
    plt.tight_layout()
    anim = animation.FuncAnimation(fig=fig, func=partial(update,lines=lines), frames=40, interval=150)
    return anim

In [ ]:
# This cell is for you to complete. It allows you to evolve a 2-field wave, once you build fns for the Fourier transfrom, 
# the dispersion relation, and the polarisation relations.

L = 10 # length of the domain
xs = np.arange(-L/2,L/2, L/256) # x in [0,L), 256 steps
t_stop = 100
delta_t = 0.1
ts = np.arange(0, t_stop, delta_t)
g = 9.8
H = 10

u = np.zeroes_like(xs) # TODO change this to reflect initial u
v = np.zeroes_like(xs) # TODO initial v
eta = np.zeroes_like(xs) # TODO initial eta

ks = # TODO find all ks, either using (r)fft.freq or coding up your own fn for slow fft
_u =  # TODO take the fourier transform of u. Make sure _u[i] corresponds to ks[i]
_v = # TODO same
_eta = # TODO same

omegas = [] # we will append to this list below with omegas for each wave mode
# eventually will have twice the length of ks because of the 2 families of wave

wave_list = {"u": [], "v": [], "eta": []} 
# a dictionary of 2 lists, each containing Fourier coeffs for each wave. Coeffs ar the same index
# in both lists correspond to the same wave

def polarisation(k, _u, _v, _eta):
    # TODO code up a function taking _u and _eta for some k
    #      and splitting each into their forward and backward parts
    return {"u": [_u_forward, _u_backward], "eta": [_eta_forward, _eta_backward], "v": [_v_forward, _v_backward]}

def calculate_omegas(k):
    # TODO for each k, calculate the two omegas
    return [forward_omega, backward_omega]



for i, k in enumerate(ks): # for each k
    k_waves =   polarisation(k, _u[i], _eta[i], _v[i])
    for key in wave_list.keys():
        wave_list[key] += k_waves[key] # appending, for each field, a list of length 2 (forward, backward)
                                       # to the existing lists
    
    omegas += calculate_omegas(k)      # appending a list of length 2 (forward, backward)

display(animate_1D(ks, xs, wave_list, omegas)) # makes and shows the animation
plt.close()

### Exercise 6

#### 6.1 Poincaré waves

(a)  Consider waves that are long with respect to the Rossby radius. Going back to the PDE, show that in this limit we approach an inertial oscillation. Show that in the opposite case (short waves) the behaviour asymptotes to that of nonrotating waves.

(b) Calculate analytically the phase and group velocities and discuss results in the long and short wave limits. You should find that these waves are *very dispersive* in the long wave limit, e.g., with ${\bf c}_g$ small and ${\bf c}_p$ large. 
		
(c) Pick either the positive or the negative $\omega$
from the dispersion relation, and visualise dispersion for a wavepacket,
varying $k L_d$ is, say, 0.5,1,2. To visualize the dispersion, you can make movies and Hovmöller diagrams. You will have to modify the same simulation and animation code as before, though with 3 fields (including $v$) even in 1D. Note that the $f$-plane model is 2D in the sense that there is a nonzero $y$-velocity, $v$; but this velocity does not *vary* along $y$.

(e) Sketch out particle (or column) trajectories for these waves in different limits.



#### 6.2 Rossby Adjustment: Dam break

(a) Write a `polarisation` and a `calculate_omegas` function for all *three* modes in 1D.  Thus, we can write a 1D polarisation relation, and make graphs of the fields $\eta(x)$, $u(x)$, $v(x)$ (and others) at any time $t$, or make an animation. Try replacing the parameters $g$ and $H$ with a single parameter $L_d=\sqrt{gH}$, because that will simplify the analysis (also in more general situations than shallow water, there are generalised notions of $L_d$ with different formulae).

(b) Analyse the wave trains that emerge from the square wave in either direction. Do they change in character if you add some initially nonzero $u$? or if you set $\eta$ to 0 and have a jump solely in $u$? Is it only the transients that change, or does the final waveform change as well? 

(c) Recall the exercise on Fourier transforms of a square wave. Now see how the errors that arise from the imperfect Fourier representation of the discontinuity propagate away. If you improve the resolution decrease $\Delta x$ and thus increase $k_{\rm max}$, does the error go away? What about if you use a $\tanh$ wave as an approximation to a square wave - are the spurious waves of the discontinuity rectified?

(d) Add fields to the simulation corresponding to $\zeta:=\partial_x v$, $\delta:=\partial_x u$ and the linearised potential vorticity $q:=\zeta-f\eta/H$. Add in a visual of the real PV $$Q:=\frac{f+\zeta}{H+\eta}$$
(which cannot be calculated in Fourier space because it's nonlinear). Explain the discrepancies between the linearised and full PV in the simulation.

#### 6.2 Gaussian initial conditions

Rinse and repeat with a Gaussian in one of these fields, varying its width. You needn't do every exercise, just explore what you find interesting. In particular, try cases where the width of the Gaussian is (a) smaller, (b) comparable or (c) larger than $L_d$. Comment on changes in the transients, and in the time for adjustment.


#### 6.3

Show that the eigenvectors derived in the pdf are orthogonal to one another $-$ e.g., such that 
${\bm \Psi}_+{\bm \Psi}_-^* = 0$,
${\bm \Psi}_+{\bm \Psi}_0^* = 0$, etc. Here, $^*$ denotes complex conjugate.
How would you normalize the eigenvectors such that 
${\bm \Psi}_0{\bm \Psi}_0^* = 1$,
${\bm \Psi}_+{\bm \Psi}_+^* = 1$,
${\bm \Psi}_-{\bm \Psi}_-^* = 1$?

#### 6.4 Extensions: harder problems! (optional)

(a) Attempt to write the polarisation relations from the eigenvector-eigenvalue perspective. Feel free to make use of numpy's linear algebra functionality, including matrix multiplication, finding eigenvectors, etc.

(b) Attempt a 2D adjustment problem. You will need the generalised ($l \neq 0$) relations derived via matrix algebra to do this.

(c) Try varying some other parameter of the problem. For instance, try a forced -dissipative version of the problem. You can imagine some precipitation or winds adding $\eta$ or $u$ to the shallow water layer as "forcing"; the "dissipation" is a damping term that kills off each sinusoid in $u$ or $v$ proportionately to its second derivative. Or try a 2-layer version of the problem. Keep everything linear.